# Bronze → Silver

Tipagem, padronização de chaves, deduplicação e regras de qualidade de cada entidade da Bronze. Registros que violam alguma regra vão para `tb_quarentena`.

## 1. Parâmetros e funções auxiliares

In [0]:
from pyspark.sql import functions as F, Window

dbutils.widgets.text("catalogo", "estoque_hospitalar", "Catálogo")
CATALOGO = dbutils.widgets.get("catalogo").strip()
spark.sql(f"USE CATALOG {CATALOGO}")

METADADOS = ["_arquivo_origem", "_dt_ingestao"]
QUARENTENA = []


def minusculas(df):
    return df.toDF(*[c.lower() for c in df.columns])


def ultimo_snapshot(tabela):
    # Cadastros chegam completos a cada extração; vale só o arquivo mais recente
    df = minusculas(spark.table(f"bronze.tb_{tabela}"))
    ultimo = df.agg(F.max("_dt_modificacao_arquivo")).first()[0]
    return df.filter(F.col("_dt_modificacao_arquivo") == ultimo)


def deduplicar(df, chaves):
    # _arquivo_origem desempata versões ingeridas na mesma carga
    janela = Window.partitionBy(*chaves).orderBy(F.col("_dt_ingestao").desc(), F.col("_arquivo_origem").desc())
    return df.withColumn("_ordem", F.row_number().over(janela)).filter("_ordem = 1").drop("_ordem")


def aplicar_regras(df, tabela, regras):
    # Gera a lista de regras violadas por linha; quem tiver alguma vai para a quarentena
    falhas = F.filter(
        F.array(*[F.when(~F.coalesce(F.expr(cond), F.lit(False)), F.lit(nome)) for nome, cond in regras.items()]),
        lambda x: x.isNotNull(),
    )
    df = df.withColumn("_falhas", falhas)
    colunas = [c for c in df.columns if c != "_falhas"]
    QUARENTENA.append(
        df.filter(F.size("_falhas") > 0).select(
            F.lit(tabela).alias("tabela"),
            F.col("_falhas").alias("regras_violadas"),
            F.to_json(F.struct(*colunas)).alias("registro"),
        )
    )
    return df.filter(F.size("_falhas") == 0).drop("_falhas")


def gravar(df, tabela, comentario):
    (
        df.withColumn("_dt_processamento", F.current_timestamp())
        .write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"silver.tb_{tabela}")
    )
    spark.sql(f"COMMENT ON TABLE silver.tb_{tabela} IS '{comentario}'")
    print(f"silver.tb_{tabela:<26} {spark.table(f'silver.tb_{tabela}').count():>12,}")

## 2. Cadastros

In [0]:
material = ultimo_snapshot("material").select(
    F.col("id_material_hospital").cast("int").alias("cd_material"),
    F.col("id_item").cast("bigint"),
    F.trim("ds_material_hospital").alias("ds_material"),
    (F.col("ie_ativo") == "S").alias("fl_ativo"),
    F.col("cd_grupo_material").cast("int").alias("cd_grupo"),
    F.trim("ds_grupo_material").alias("ds_grupo"),
    F.col("cd_subgrupo_material").cast("int").alias("cd_subgrupo"),
    F.trim("ds_subgrupo_material").alias("ds_subgrupo"),
    F.col("cd_classe_material").cast("int").alias("cd_classe"),
    F.trim("ds_classe_material").alias("ds_classe"),
    F.col("cd_unidade_medida_compra").alias("cd_unidade_compra"),
    F.col("ds_unidade_medida_compra").alias("ds_unidade_compra"),
    F.col("cd_unidade_medida_estoque").alias("cd_unidade_estoque"),
    F.col("ds_unidade_medida_estoque").alias("ds_unidade_estoque"),
    F.col("cd_unidade_medida_consumo").alias("cd_unidade_consumo"),
    F.col("ds_unidade_medida_consumo").alias("ds_unidade_consumo"),
    F.col("qt_conv_compra_estoque").cast("double"),
    F.col("qt_conv_estoque_consumo").cast("double"),
    F.to_timestamp("dt_cadastramento", "yyyy-MM-dd HH:mm:ss.SSS").alias("dt_cadastramento"),
    *METADADOS,
)
material = aplicar_regras(material, "material", {
    "material_sem_codigo": "cd_material IS NOT NULL",
    "material_sem_grupo": "cd_grupo IS NOT NULL",
    "conversao_invalida": "qt_conv_compra_estoque > 0 AND qt_conv_estoque_consumo > 0",
})
gravar(material, "material", "Cadastro de materiais, versão mais recente")

silver.tb_material                            871


In [0]:
operacao = ultimo_snapshot("operacao_estoque").select(
    F.col("cd_operacao_estoque").cast("int").alias("cd_operacao"),
    F.trim("ds_operacao_estoque").alias("ds_operacao"),
    F.trim("ds_operacao").alias("ds_tipo_operacao"),
    (F.trim("ds_operacao") == "Entrada").alias("fl_entrada"),
    # Sinal aplicado à quantidade para calcular o saldo na Gold
    F.when(F.trim("ds_operacao") == "Entrada", 1).otherwise(-1).alias("nr_sinal_saldo"),
    *METADADOS,
)
gravar(operacao, "operacao_estoque", "Operações de estoque com o sinal de efeito no saldo")

local = ultimo_snapshot("local_estoque").select(
    F.col("cd_local_estoque").cast("int"),
    F.trim("ds_local_estoque").alias("ds_local_estoque"),
    F.col("cd_estabelecimento").cast("int"),
    F.trim("ds_tipo_local").alias("ds_tipo_local"),
    F.col("cd_centro_custo").cast("int"),
    (F.col("ie_situacao") == "A").alias("fl_ativo"),
    *METADADOS,
)
gravar(local, "local_estoque", "Locais de estoque com o nome atual")

# O cadastro de locais é a única fonte com código e nome do estabelecimento
estabelecimento = ultimo_snapshot("local_estoque").select(
    F.col("cd_estabelecimento").cast("int"),
    F.trim("ds_estabelecimento").alias("ds_estabelecimento"),
).dropDuplicates(["cd_estabelecimento"])
gravar(estabelecimento, "estabelecimento", "Estabelecimentos da rede")

fornecedor = ultimo_snapshot("fornecedor").select(
    F.col("cd_fornecedor").cast("int"),
    F.trim("ds_razao_social").alias("ds_razao_social"),
    F.trim("nm_fantasia").alias("nm_fantasia"),
    F.trim("ds_municipio").alias("ds_municipio"),
    F.upper(F.trim("sg_uf")).alias("sg_uf"),
    F.col("qt_dias_entrega").cast("int"),
    (F.col("ie_situacao") == "A").alias("fl_ativo"),
    F.to_date("dt_cadastro").alias("dt_cadastro"),
    *METADADOS,
)
gravar(fornecedor, "fornecedor", "Fornecedores")

silver.tb_operacao_estoque                     13
silver.tb_local_estoque                        18
silver.tb_estabelecimento                       5
silver.tb_fornecedor                           47


## 3. tb_movimento_estoque

| Tratamento | Regra |
|---|---|
| `cd_conta_contabil` | Recuperada de `_rescued_data` nos arquivos em que veio como número, sem o sufixo `.0` |
| Ano digitado errado | Data reconstruída com o ano da competência, marcada em `fl_data_movimento_corrigida` |
| Lançamento na competência seguinte | Marcado em `fl_competencia_seguinte` |
| Colunas descartadas | `vl_estoque` e `vl_movimento_2` (cópias), `nr_lote_contabil` (vazia), `id_item` (derivada) |
| Estornos | `cd_acao = 2` com quantidade negativa |

In [0]:
conta_resgatada = (
    F.get_json_object("_rescued_data", "$.cd_conta_contabil")
    .cast("double").cast("decimal(20,0)").cast("string")
)
data_reconstruida = F.expr(
    "make_timestamp(year(dt_competencia), month(dt_movimento_original), day(dt_movimento_original), "
    "hour(dt_movimento_original), minute(dt_movimento_original), second(dt_movimento_original))"
)

mov_base = (
    deduplicar(minusculas(spark.table("bronze.tb_movimento_estoque")), ["nr_movimento_estoque"])
    .withColumn("dt_movimento_original", F.to_timestamp("dt_movimento_estoque", "yyyy-MM-dd HH:mm:ss.SSS"))
    .withColumn("dt_competencia", F.to_date(F.substring("dt_referencia", 1, 10)))
    # 1 mês de diferença é lançamento na competência seguinte; mais que isso é ano digitado errado
    .withColumn("_dif_meses", F.months_between("dt_competencia", F.trunc("dt_movimento_original", "month")))
    .withColumn("fl_competencia_seguinte", F.col("_dif_meses") == 1)
    .withColumn("fl_data_movimento_corrigida", F.abs("_dif_meses") > 1)
    .withColumn("dt_movimento", F.when(F.col("fl_data_movimento_corrigida"), data_reconstruida)
                                 .otherwise(F.col("dt_movimento_original")))
    .withColumn("cd_conta_contabil", F.coalesce(F.regexp_replace("cd_conta_contabil", r"\.0$", ""), conta_resgatada))
)

In [0]:
movimento = mov_base.select(
    F.col("nr_movimento_estoque").cast("bigint"),
    "dt_movimento",
    "dt_movimento_original",
    "dt_competencia",
    F.col("cd_material").cast("int"),
    F.col("cd_local_estoque").cast("int"),
    F.col("cd_centro_custo").cast("int"),
    F.col("cd_operacao").cast("int"),
    F.col("cd_acao").cast("int"),
    (F.col("cd_acao") == 2).alias("fl_estorno"),
    F.col("nr_atendimento").cast("bigint"),
    F.col("nr_seq_lote_fornec").cast("bigint").alias("nr_seq_lote"),
    "cd_conta_contabil",
    F.col("qt_estoque").cast("double"),
    F.col("qt_consumo").cast("double"),
    F.col("vl_movimento").cast("double"),
    F.col("vl_consumo").cast("double"),
    "fl_competencia_seguinte",
    "fl_data_movimento_corrigida",
    *METADADOS,
)
movimento = aplicar_regras(movimento, "movimento_estoque", {
    "chave_nula": "nr_movimento_estoque IS NOT NULL",
    "material_nulo": "cd_material IS NOT NULL",
    "local_nulo": "cd_local_estoque IS NOT NULL",
    "operacao_nula": "cd_operacao IS NOT NULL",
    "data_nula": "dt_movimento IS NOT NULL AND dt_competencia IS NOT NULL",
    "quantidade_nula": "qt_estoque IS NOT NULL AND qt_estoque <> 0",
    "sinal_estorno": "(cd_acao = 2 AND qt_estoque < 0) OR (cd_acao = 1 AND qt_estoque > 0)",
    "conta_contabil_nula": "cd_conta_contabil IS NOT NULL",
})
gravar(movimento, "movimento_estoque", "Movimentações de estoque tipadas e deduplicadas")

silver.tb_movimento_estoque             2,656,470


## 4. Dimensões derivadas da movimentação

O ERP não exporta cadastro de centro de custo nem o histórico de nomes dos locais.

In [0]:
# Estabelecimento pelo código do local que atende o centro de custo, não pela descrição
local_estabelecimento = spark.table("silver.tb_local_estoque").select("cd_local_estoque", "cd_estabelecimento")
centro_custo = (
    mov_base.groupBy(F.col("cd_centro_custo").cast("int").alias("cd_centro_custo"))
    .agg(
        F.trim(F.max_by("ds_centro_custo", "dt_movimento")).alias("ds_centro_custo"),
        F.max_by(F.col("cd_local_estoque").cast("int"), "dt_movimento").alias("cd_local_estoque"),
    )
    .join(local_estabelecimento, "cd_local_estoque", "left")
    .select("cd_centro_custo", "ds_centro_custo", "cd_estabelecimento")
)
gravar(centro_custo, "centro_custo", "Centros de custo derivados da movimentação")

# SCD tipo 2: cada nome vale até a véspera do primeiro uso do nome seguinte
janela_local = Window.partitionBy("cd_local_estoque").orderBy("dt_inicio_vigencia")
local_historico = (
    mov_base.groupBy(
        F.col("cd_local_estoque").cast("int").alias("cd_local_estoque"),
        F.trim("ds_local_estoque").alias("ds_local_estoque"),
    )
    .agg(F.to_date(F.min("dt_movimento")).alias("dt_inicio_vigencia"))
    .withColumn("dt_fim_vigencia", F.date_sub(F.lead("dt_inicio_vigencia").over(janela_local), 1))
    .withColumn("dt_inicio_vigencia", F.when(F.row_number().over(janela_local) == 1, F.lit("1900-01-01").cast("date"))
                                       .otherwise(F.col("dt_inicio_vigencia")))
    .withColumn("fl_vigente", F.col("dt_fim_vigencia").isNull())
)
gravar(local_historico, "local_estoque_historico", "Histórico de nomes dos locais de estoque (SCD2)")

silver.tb_centro_custo                         49
silver.tb_local_estoque_historico              20


## 5. Notas fiscais, lotes e saldos

In [0]:
nota_fiscal = deduplicar(minusculas(spark.table("bronze.tb_nota_fiscal")), ["nr_sequencia"]).select(
    F.col("nr_sequencia").cast("bigint").alias("nr_seq_nota_fiscal"),
    F.col("nr_nota_fiscal").cast("string"),
    F.col("cd_serie_nf").cast("string"),
    F.col("cd_estabelecimento").cast("int"),
    F.col("cd_fornecedor").cast("int"),
    F.to_timestamp("dt_emissao", "yyyy-MM-dd HH:mm:ss").alias("dt_emissao"),
    F.to_timestamp("dt_entrada_saida", "yyyy-MM-dd HH:mm:ss").alias("dt_entrada"),
    F.col("vl_mercadoria").cast("decimal(18,2)"),
    F.col("vl_frete").cast("decimal(18,2)"),
    F.col("vl_total_nota").cast("decimal(18,2)"),
    *METADADOS,
)
nota_fiscal = aplicar_regras(nota_fiscal, "nota_fiscal", {
    "chave_nula": "nr_seq_nota_fiscal IS NOT NULL",
    "fornecedor_nulo": "cd_fornecedor IS NOT NULL",
    "total_divergente": "abs(vl_total_nota - vl_mercadoria - vl_frete) <= 0.01",
    "entrada_antes_emissao": "to_date(dt_entrada) >= to_date(dt_emissao)",
})
gravar(nota_fiscal, "nota_fiscal", "Cabeçalho das notas fiscais de entrada")

nota_fiscal_item = deduplicar(minusculas(spark.table("bronze.tb_nota_fiscal_item")), ["nr_sequencia_nf", "nr_item_nf"]).select(
    F.col("nr_sequencia_nf").cast("bigint").alias("nr_seq_nota_fiscal"),
    F.col("nr_item_nf").cast("int"),
    F.col("cd_material").cast("int"),
    F.col("cd_local_estoque").cast("int"),
    F.col("nr_seq_lote_fornec").cast("bigint").alias("nr_seq_lote"),
    F.col("cd_unidade_medida_compra").alias("cd_unidade_compra"),
    F.col("qt_item_nf").cast("double"),
    F.col("qt_item_estoque").cast("double"),
    F.col("vl_unitario_item_nf").cast("double").alias("vl_unitario_compra"),
    F.col("vl_total_item_nf").cast("decimal(18,2)").alias("vl_total_item"),
    # A nota pode vir em caixa ou pacote; o custo é levado para a unidade de estoque
    (F.col("vl_total_item_nf") / F.col("qt_item_estoque")).alias("vl_unitario_estoque"),
    *METADADOS,
)
nota_fiscal_item = aplicar_regras(nota_fiscal_item, "nota_fiscal_item", {
    "chave_nula": "nr_seq_nota_fiscal IS NOT NULL AND nr_item_nf IS NOT NULL",
    "quantidade_invalida": "qt_item_nf > 0 AND qt_item_estoque > 0",
    "valor_negativo": "vl_total_item >= 0",
})
gravar(nota_fiscal_item, "nota_fiscal_item", "Itens das notas fiscais de entrada")

silver.tb_nota_fiscal                       8,932
silver.tb_nota_fiscal_item                 24,151


In [0]:
lote = deduplicar(minusculas(spark.table("bronze.tb_lote_fornecedor")), ["nr_sequencia"]).select(
    F.col("nr_sequencia").cast("bigint").alias("nr_seq_lote"),
    F.col("cd_material").cast("int"),
    F.trim("ds_lote_fornec").alias("ds_lote_fornecedor"),
    F.to_date("dt_fabricacao").alias("dt_fabricacao"),
    F.to_date("dt_validade").alias("dt_validade"),
    F.col("cd_fornecedor").cast("int"),
    F.col("nr_seq_nota_fiscal").cast("bigint"),
    F.col("dt_validade").isNotNull().alias("fl_controla_validade"),
    *METADADOS,
)
lote = aplicar_regras(lote, "lote_fornecedor", {
    "chave_nula": "nr_seq_lote IS NOT NULL",
    "validade_antes_fabricacao": "dt_validade IS NULL OR dt_fabricacao IS NULL OR dt_validade > dt_fabricacao",
})
gravar(lote, "lote_fornecedor", "Lotes de fornecedor com fabricação e validade")

saldo = deduplicar(
    minusculas(spark.table("bronze.tb_saldo_estoque")),
    ["dt_mesano_referencia", "cd_local_estoque", "cd_material", "nr_seq_lote_fornec"],
).select(
    F.to_date("dt_mesano_referencia").alias("dt_competencia"),
    # O saldo é o fechamento do mês: a posição vale para o último dia da competência
    F.last_day(F.to_date("dt_mesano_referencia")).alias("dt_posicao"),
    F.col("cd_estabelecimento").cast("int"),
    F.col("cd_local_estoque").cast("int"),
    F.col("cd_material").cast("int"),
    F.col("nr_seq_lote_fornec").cast("bigint").alias("nr_seq_lote"),
    F.col("qt_estoque").cast("double"),
    F.col("vl_custo_medio").cast("double"),
    F.col("vl_estoque").cast("decimal(18,2)"),
    *METADADOS,
)
saldo = aplicar_regras(saldo, "saldo_estoque", {
    "chave_nula": "dt_competencia IS NOT NULL AND cd_material IS NOT NULL AND cd_local_estoque IS NOT NULL",
    "saldo_negativo": "qt_estoque >= 0",
})
gravar(saldo, "saldo_estoque", "Saldo de fechamento mensal por item, local e lote")

silver.tb_lote_fornecedor                  27,536
silver.tb_saldo_estoque                    95,519


## 6. Quarentena e conferência

In [0]:
quarentena = QUARENTENA[0]
for df in QUARENTENA[1:]:
    quarentena = quarentena.unionByName(df)
gravar(quarentena, "quarentena", "Registros que violaram regras de qualidade da silver")

resumo = (
    spark.table("silver.tb_quarentena")
    .select("tabela", F.explode("regras_violadas").alias("regra"))
    .groupBy("tabela", "regra").count()
    .orderBy("tabela", "regra")
)
if resumo.isEmpty():
    print("Nenhum registro violou as regras de qualidade")
else:
    display(resumo)

silver.tb_quarentena                            0
Nenhum registro violou as regras de qualidade


In [0]:
display(
    spark.table("silver.tb_movimento_estoque").agg(
        F.count("*").alias("linhas"),
        F.countDistinct("nr_movimento_estoque").alias("chaves_distintas"),
        F.sum(F.col("fl_estorno").cast("int")).alias("estornos"),
        F.sum(F.col("fl_competencia_seguinte").cast("int")).alias("competencia_seguinte"),
        F.sum(F.col("fl_data_movimento_corrigida").cast("int")).alias("data_corrigida"),
        F.sum(F.col("cd_conta_contabil").isNull().cast("int")).alias("conta_nula"),
        F.countDistinct("cd_conta_contabil").alias("contas_distintas"),
        F.min("dt_movimento").alias("primeiro_movimento"),
        F.max("dt_movimento").alias("ultimo_movimento"),
    )
)

linhas,chaves_distintas,estornos,competencia_seguinte,data_corrigida,conta_nula,contas_distintas,primeiro_movimento,ultimo_movimento
2656470,2656470,1211,1110,87,0,45,2024-10-01T00:01:06.000Z,2026-09-30T23:59:50.000Z
